1. 문서의 내용으 읽는다
2. 문서를 쪼갠다.
  - 토큰 수 초과로 답변을 생성하지 못할 수 있고
  - 문서가 길면(Input) 답변 생성이 오래걸림
3. 쪼갠 문서를 임배딩 -> 백터 데이터베이스에 저장
4. 질문이 있을 때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
# %pip install --upgrade --quiet  docx2txt langchain-community


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 문서 쪼개기 Text Splitter

# Character Text Splitter : 구분자를 하나만 넣을 수 있다.
# Recursive Character Text Splitter : 구분자를 여러개 넣을 수 있다.

# %pip install -qU langchain-text-splitters







Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # 문서를 쪼갤 때 하나의 청크가 가질 수 있는 청크의 수
    chunk_size=1500,
    # 문서를 곂치는 설정
    chunk_overlap=200
)

loader = Docx2txtLoader("./tax.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
len(document_list)

193

In [5]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings


load_dotenv()

embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')


In [6]:
# %pip install langchain-chroma

In [8]:
# %pip install --upgrade --quiet \
#     langchain-pinecone \
#     langchain-openai \
#     langchain \
#     pinecone-notebooks
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-index'
pinecone_api_key= os.environ.get("PINECONE_API_KEY")
pinecone_api_key

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding=embedding, index_name=index_name)


In [9]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')



In [11]:
prompt = f"""
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

Question : {query}
"""

In [12]:
ai_message = llm.invoke(prompt)

ai_message.content




'연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 아래의 단계를 따를 수 있습니다.\n\n1. **총급여액 및 비과세소득 제외**: 연간 총급여 5천만원 중에서 비과세소득(만약 있다면)을 우선적으로 제외합니다. 비과세소득의 예로는 일부 식사비, 경조사비 등이 있을 수 있습니다. 하지만, 일반적인 경우에는 이러한 금액이 크지 않기 때문에 제외하지 않더라도 큰 차이는 없을 수 있습니다.\n\n2. **근로소득공제**: 총급여액에서 근로소득공제를 합니다. 근로소득공제는 일반적으로 단계별로 계산되며, 과세표준을 낮춰주는 역할을 합니다. 예를 들어, 2023년도 기준으로는 대략 다음과 같이 계산됩니다:\n   - 5천만원까지는 72만원 + (총급여액 - 4600만원) × 35%\n\n3. **과세표준 계산**: 총급여액에서 근로소득공제를 제외한 금액이 과세표준이 됩니다.\n\n4. **세율 적용**: 과세표준에 따라 누진세율이 적용됩니다. 2023년도 기준으로는 대략 다음과 같은 세율이 적용됩니다:\n   - 1,200만원 이하: 6%\n   - 1,200만원 초과 ~ 4,600만원 이하: 15%\n   - 4,600만원 초과 ~ 8,800만원 이하: 24%\n   - 8,800만원 초과 ~ 1억5천만원 이하: 35%\n\n5. **산출세액 계산**: 각 구간별 세율을 적용하여 산출세액을 계산합니다.\n\n6. **세액공제 적용**: 근로소득세액공제 등 여러 공제를 적용합니다.\n\n7. **결정세액 계산**: 산출세액에서 공제액을 빼서 최종적으로 결정세액을 계산합니다.\n\n정확한 세액은 공제와 같은 여러 요인에 따라 달라질 수 있습니다. 따라서, 대체로 계산기를 사용하거나 국세청의 연말정산 서비스를 활용하여 정확한 금액을 확인하는 것이 좋습니다. 기본적으로 위의 과정을 거쳐 대략적인 소득세를 계산할 수 있습니다.'

In [12]:
# %pip install -U langchain langchain_classic langchainhub --quiet

In [13]:
from langchain_classic import hub

prompt = hub.pull('rlm/rag-prompt')


In [14]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [14]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={'prompt': prompt}
)

In [15]:
ai_message = qa_chain.invoke({'query': query})

In [16]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '죄송하지만 주어진 문맥 정보에서는 연봉 5천만 원에 대한 정확한 소득세 계산 정보를 포함하지 않습니다. 한국의 소득세는 소득 구간별로 다른 세율이 적용되며, 각종 공제와 혜택에 따라 달라질 수 있습니다. 소득세 계산에 관한 최근 정보를 위해 정부 웹사이트 또는 세무 전문가의 상담을 추천드립니다.'}